# 2. SupportSense — Support Topic Segmentation
### Creating business segments from support questions

This notebook follows the second reference notebook's core pattern: define segments, create analysis variables, summarize records at the entity level, assign interpretable labels, and compare the resulting segments.

**Scope:** the current SupportSense repository contains a curated synthetic knowledge base and a synthetic evaluation-question set, not historical customer or ticket records. We therefore segment *benchmark questions* by topic and answerability. These groups can help prioritize retrieval testing, but they are not customer segments and must not be presented as real customer behavior.

## 1. Load and inspect the evaluation data

The data is read from the project JSON file. Path resolution supports running the notebook from the repository root or from `notebooks/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), Path.cwd().parent]
             if (p / "data" / "evaluation_questions.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or its notebooks/ folder.")

with (ROOT / "data" / "evaluation_questions.json").open(encoding="utf-8") as f:
    questions = pd.DataFrame(json.load(f))
with (ROOT / "data" / "company_docs.json").open(encoding="utf-8") as f:
    docs = pd.DataFrame(json.load(f))

print(f"Questions: {len(questions)}")
print(f"Knowledge-base documents: {len(docs)}")
display(questions.head())

## 2. Define interpretable support-topic groups

The mapping below links each benchmark question to the topic of its expected source document. Questions with no expected source are retained as `Needs verification / escalation`. This is an explicit business mapping, not an unsupervised model.

In [ ]:
topic_map = {
    "KB-001": "Account access & security",
    "KB-002": "Billing & subscriptions",
    "KB-003": "Billing & subscriptions",
    "KB-004": "Account access & security",
    "KB-005": "API & technical support",
    "KB-006": "Data management",
    "KB-007": "Service reliability",
    "KB-008": "Account access & security",
    "KB-009": "Billing & subscriptions",
    "KB-010": "Account access & security",
}
doc_titles = docs.set_index("doc_id")["title"].to_dict()
doc_topics = docs["doc_id"].map(topic_map)
doc_topic_lookup = dict(zip(docs["doc_id"], doc_topics))

seg = questions.copy()
seg["expected_title"] = seg["expected_doc"].map(doc_titles)
seg["topic_segment"] = seg["expected_doc"].map(doc_topic_lookup).fillna("Needs verification / escalation")
seg["answerability_segment"] = np.where(
    seg["answerable"].astype(bool),
    "Answerable from current knowledge base",
    "Needs verification / escalation"
)
seg["question_words"] = seg["question"].astype(str).str.split().str.len()
seg["expected_terms_count"] = seg["expected_terms"].apply(lambda x: len(x) if isinstance(x, list) else 0)

display(seg[["question_id", "question", "expected_title", "topic_segment",
             "answerability_segment", "question_words", "expected_terms_count"]])

## 3. Create benchmark-level RFM-style variables

The reference notebook uses Recency, Frequency, and Monetary (RFM) features because its rows represent customer rental activity. SupportSense has no customer transactions or timestamps, so RFM would be inappropriate here.

Instead, we create three *benchmark prioritization indicators*:
- **Coverage frequency:** how many benchmark questions map to the same topic.
- **Question complexity proxy:** word count of the question (not a validated difficulty score).
- **Verification need:** whether the question is labelled unanswerable from current documents.

These variables describe the benchmark design, not actual customer usage.

In [ ]:
topic_counts = seg["topic_segment"].value_counts()
seg["topic_question_count"] = seg["topic_segment"].map(topic_counts)
seg["verification_needed"] = ~seg["answerable"].astype(bool)

# A transparent heuristic score for review ordering; it is not a trained prediction.
seg["review_priority_score"] = (
    seg["verification_needed"].astype(int) * 2
    + (seg["question_words"] >= seg["question_words"].median()).astype(int)
    + (seg["expected_terms_count"] == 0).astype(int)
)

display(seg[["question_id", "topic_segment", "topic_question_count",
             "question_words", "verification_needed", "review_priority_score"]].sort_values(
                 ["review_priority_score", "question_words"], ascending=False))

## 4. Summarize the segments

Compare question counts and answerability across topic groups. Small counts are expected because this is a compact benchmark of 12 questions; percentages should not be generalized to production support.

In [ ]:
segment_summary = (
    seg.groupby("topic_segment", dropna=False)
       .agg(
           question_count=("question_id", "nunique"),
           mean_question_words=("question_words", "mean"),
           answerable_count=("answerable", lambda s: int(s.astype(bool).sum())),
           verification_count=("verification_needed", "sum"),
           mean_review_priority=("review_priority_score", "mean")
       )
       .reset_index()
       .sort_values("question_count", ascending=False)
)
segment_summary["mean_question_words"] = segment_summary["mean_question_words"].round(1)
segment_summary["mean_review_priority"] = segment_summary["mean_review_priority"].round(2)
display(segment_summary)

In [ ]:
plot_df = segment_summary.sort_values("question_count")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["topic_segment"], plot_df["question_count"])
ax.set_title("Benchmark questions by topic segment")
ax.set_xlabel("Number of benchmark questions")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(segment_summary))
ax.bar(x - 0.18, segment_summary["answerable_count"], width=0.36, label="Answerable")
ax.bar(x + 0.18, segment_summary["verification_count"], width=0.36, label="Needs verification")
ax.set_xticks(x)
ax.set_xticklabels(segment_summary["topic_segment"], rotation=25, ha="right")
ax.set_title("Answerability mix within benchmark segments")
ax.set_ylabel("Question count")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Create simple segment labels for review

The labels below are intended for a human review queue:
- **Retrieval validation:** answerable question with an expected source.
- **Grounding / abstention test:** question labelled unanswerable.
- **Review mapping:** expected source or topic mapping needs inspection.

This is a rule-based grouping, not a machine-learning classifier.

In [ ]:
def assign_review_segment(row):
    if not bool(row["answerable"]):
        return "Grounding / abstention test"
    if pd.isna(row["expected_doc"]) or pd.isna(row["expected_title"]):
        return "Review mapping"
    return "Retrieval validation"

seg["review_segment"] = seg.apply(assign_review_segment, axis=1)
review_summary = (
    seg.groupby("review_segment")
       .agg(
           questions=("question_id", "nunique"),
           avg_words=("question_words", "mean"),
           topics=("topic_segment", "nunique")
       )
       .reset_index()
)
review_summary["avg_words"] = review_summary["avg_words"].round(1)
display(review_summary)
display(seg[["question_id", "question", "review_segment", "topic_segment", "expected_doc"]])

## 6. Identify where the benchmark is thin

A segment with only one question provides little evidence about retrieval quality for that topic. This cell identifies segments with fewer than two examples and lists questions that need human review. The threshold is a practical flag, not a statistical adequacy rule.

In [ ]:
thin_segments = segment_summary[segment_summary["question_count"] < 2]
print(f"Topic segments with fewer than 2 benchmark questions: {len(thin_segments)}")
display(thin_segments)

print("Questions requiring grounding / abstention checks:")
display(seg.loc[seg["review_segment"] == "Grounding / abstention test",
                ["question_id", "question", "expected_doc", "expected_terms"]])

## 7. Export the segmented benchmark

The export is useful for the next notebook and for manual audit. It contains derived analysis fields but does not change the original source JSON files.

In [ ]:
output_path = ROOT / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)
export_cols = [
    "question_id", "question", "expected_doc", "expected_title", "topic_segment",
    "answerable", "answerability_segment", "question_words", "expected_terms_count",
    "topic_question_count", "verification_needed", "review_priority_score", "review_segment"
]
seg[export_cols].to_csv(output_path / "segmented_evaluation_questions.csv", index=False)
print("Saved:", output_path / "segmented_evaluation_questions.csv")

## 8. Findings and limitations

### What we can conclude
- The notebook groups the current benchmark into interpretable support topics.
- Unanswerable examples are explicitly separated for grounding and abstention evaluation.
- Topic groups with very few questions should receive additional test cases before strong conclusions are drawn.

### What we cannot conclude
- These groups are not customer segments; no real customer-level records are available.
- Question count is not the same as issue frequency in a live support organization.
- The review-priority score is a transparent heuristic, not a validated prediction of risk or difficulty.
- A larger benchmark, reviewed by subject-matter experts, is needed before comparing topic-level model performance.

**Next notebook:** *SupportSense — Retrieval & RAG Modeling*, which evaluates whether the correct source documents are retrieved and whether answers remain grounded.